# CC3092 – Deep Learning · Proyecto 2
## Detección secuencial de lavado de dinero en transacciones IBM AML

**Curso:** CC3092 – Deep Learning  
**Fecha:** [COMPLETAR]

Este notebook implementa un sistema de detección en dos etapas:

1. **Etapa A:** un LSTM Encoder–Decoder aprende a reconstruir únicamente secuencias normales. El error de reconstrucción se convierte en score de anomalía.
2. **Etapa B:** un clasificador reutiliza el encoder anterior, incorpora atención de producto punto escalado y estima la probabilidad de lavado.

Se incluye una línea base entrenada desde cero, combinación de señales, ablación, interpretabilidad por transacción y exportación de artefactos para el MVP.

> El notebook no contiene métricas inventadas. Debe ejecutarse con los archivos de datos para producir los resultados finales.

## Bloque 0 — Configuración y reproducibilidad

El modo rápido permite verificar el pipeline. Para la ejecución final se debe usar `MODO_RAPIDO = False`. El notebook está preparado para ejecutarse localmente desde la raíz del proyecto y utiliza rutas relativas, de modo que la carpeta completa pueda moverse o compartirse sin modificar rutas absolutas.

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
import copy, csv, json, math, os, random, time, warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

@dataclass
class Config:
    SEED: int = 42
    MODO_RAPIDO: bool = False
    DATA_DIR: str = '.'
    IBM_FILE: str = 'ibm-transactions/HI-Small_Trans.csv'
    PATTERNS_FILE: str = 'ibm-transactions/HI-Small_Patterns.txt'
    PAYSIM_FILE: str = 'paysim/PS_20174392719_1491204439457_log.csv'
    ARTIFACT_DIR: str = 'artifacts_aml'
    MIN_SEQ_LEN: int = 3
    MAX_SEQ_LEN: int = 30
    MAX_WINDOWS_PER_SENDER: int = 20
    MAX_NORMAL_AE: int = 60000
    NORMAL_TO_POSITIVE_B: int = 10
    BATCH_SIZE: int = 256
    HIDDEN_SIZE: int = 64
    LATENT_SIZE: int = 64
    ATTN_SIZE: int = 32
    DROPOUT: float = 0.20
    AE_EPOCHS: int = 8
    HEAD_EPOCHS: int = 3
    FINETUNE_EPOCHS: int = 6
    LR_AE: float = 1e-3
    LR_HEAD: float = 1e-3
    LR_ENCODER: float = 1e-4
    PATIENCE: int = 3
    NUM_WORKERS: int = 0

CFG = Config()
if CFG.MODO_RAPIDO:
    CFG.MAX_NORMAL_AE = 8000
    CFG.AE_EPOCHS = 2
    CFG.HEAD_EPOCHS = 1
    CFG.FINETUNE_EPOCHS = 2

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = (PROJECT_ROOT / CFG.DATA_DIR).resolve()
IBM_PATH = DATA_DIR / CFG.IBM_FILE
PATTERNS_PATH = DATA_DIR / CFG.PATTERNS_FILE
PAYSIM_PATH = DATA_DIR / CFG.PAYSIM_FILE
ARTIFACT_DIR = (PROJECT_ROOT / CFG.ARTIFACT_DIR).resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
T0_GLOBAL = time.time()

print('PyTorch:', torch.__version__)
print('Pandas:', pd.__version__)
print('Dispositivo:', DEVICE)
print('Raíz del proyecto:', PROJECT_ROOT)
print('Archivo IBM:', IBM_PATH)
print('Archivo de patrones:', PATTERNS_PATH)
print('Directorio de artefactos:', ARTIFACT_DIR)
print('Modo rápido:', CFG.MODO_RAPIDO)

PyTorch: 2.14.0+cpu
Pandas: 2.3.3
Dispositivo: cpu
Raíz del proyecto: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2
Archivo IBM: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\ibm-transactions\HI-Small_Trans.csv
Archivo de patrones: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\ibm-transactions\HI-Small_Patterns.txt
Directorio de artefactos: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\artifacts_aml
Modo rápido: False


## Bloque 1 — Decisión de dataset

En nuestra exploración del dataset encontramos que PaySim tiene 6,353,307 remitentes, pero 99 % de ellos posee una sola transacción. Solo 28 remitentes fraudulentos tienen dos transacciones y ninguno tiene tres. Por ello, PaySim no permite aprender comportamiento histórico real por remitente.

IBM AML sí contiene historiales: HI-Small tiene 496,999 remitentes, 3,376 remitentes positivos y percentil 90 de longitud igual a 29. Se selecciona HI-Small porque es menor que LI-Small y ofrece más casos positivos, aunque conserva un desbalance fuerte.

In [3]:
paysim_audit = pd.Series({
    'transacciones': 6_362_620, 'remitentes': 6_353_307,
    'remitentes_longitud_2_o_mas': 9_298,
    'remitentes_longitud_3_o_mas': 15,
    'remitentes_fraudulentos': 8_213,
    'fraudulentos_longitud_2_o_mas': 28,
    'fraudulentos_longitud_3_o_mas': 0
}, name='PaySim — auditoría ejecutada por el grupo')

ibm_comparison = pd.DataFrame({
    'HI-Small': [5_078_345, 5_177, 496_999, 3_376, 0.00679277, 2, 46, 92],
    'LI-Small': [6_924_049, 3_565, 681_283, 2_382, 0.00349634, 2, 46, 92]
}, index=['transacciones','transacciones_positivas','remitentes','remitentes_positivos',
          'tasa_remitentes_positivos','mediana_longitud','p95_longitud','p99_longitud'])

display(paysim_audit.to_frame())
display(ibm_comparison)
assert paysim_audit['fraudulentos_longitud_3_o_mas'] == 0
assert ibm_comparison.loc['remitentes_positivos','HI-Small'] > ibm_comparison.loc['remitentes_positivos','LI-Small']
print('Decisión documentada: HI-Small será el dataset secuencial principal.')

,PaySim — auditoría ejecutada por el grupo
transacciones,6362620
remitentes,6353307
remitentes_longitud_2_o_mas,9298
remitentes_longitud_3_o_mas,15
remitentes_fraudulentos,8213
fraudulentos_longitud_2_o_mas,28
fraudulentos_longitud_3_o_mas,0


,HI-Small,LI-Small
transacciones,5.078345e+06,6.924049e+06
transacciones_positivas,5.177000e+03,3.565000e+03
remitentes,4.969990e+05,6.812830e+05
remitentes_positivos,3.376000e+03,2.382000e+03
tasa_remitentes_positivos,6.792770e-03,3.496340e-03
mediana_longitud,2.000000e+00,2.000000e+00
p95_longitud,4.600000e+01,4.600000e+01
p99_longitud,9.200000e+01,9.200000e+01


Decisión documentada: HI-Small será el dataset secuencial principal.


### Justificación de la selección del dataset

Aunque PaySim se consideró inicialmente como dataset principal, el análisis de su estructura mostró una limitación importante para este proyecto: 6,353,307 de sus remitentes corresponden a una población en la que aproximadamente el 99 % presenta una sola transacción. Entre los 8,213 remitentes asociados con fraude, únicamente 28 poseen al menos dos transacciones y ninguno alcanza tres. Por tanto, construir secuencias históricas por remitente requeriría crear artificialmente una estructura temporal que los datos no contienen.

En contraste, IBM AML HI-Small contiene 5,078,345 transacciones correspondientes a 496,999 remitentes, de los cuales 3,376 presentan al menos una transacción etiquetada como lavado. La mediana del historial es de 2 transacciones, el percentil 90 es 29 y el percentil 95 es 46. Esta estructura permite estudiar comportamiento secuencial real dentro del dataset.

Por esta razón se seleccionó **IBM AML HI-Small como dataset principal para el modelado secuencial**. La decisión no se basa únicamente en tamaño, sino en la compatibilidad entre la estructura de los datos y el objetivo del proyecto: aprender patrones temporales de comportamiento por remitente. PaySim se conserva como referencia comparativa para justificar esta decisión.

## Bloque 2 — Carga y auditoría de HI-Small

El remitente se identifica con la clave compuesta `(From Bank, Account)`, ya que un número de cuenta aislado no representa necesariamente una identidad global. La etiqueta nunca se usa como feature.

In [4]:
assert IBM_PATH.exists(), f'No se encontró {IBM_PATH}. Ajuste CFG.DATA_DIR.'

DTYPES = {
    'From Bank':'int32', 'Account':'string', 'To Bank':'int32', 'Account.1':'string',
    'Amount Received':'float32', 'Receiving Currency':'category',
    'Amount Paid':'float32', 'Payment Currency':'category',
    'Payment Format':'category', 'Is Laundering':'int8'
}

t0 = time.time()
df = pd.read_csv(IBM_PATH, dtype=DTYPES)
df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y/%m/%d %H:%M', errors='coerce')
df['sender_id'] = df['From Bank'].astype(str) + '::' + df['Account'].astype(str)
df['dest_id'] = df['To Bank'].astype(str) + '::' + df['Account.1'].astype(str)

print(f'Filas: {len(df):,} | Columnas: {df.shape[1]} | Carga: {time.time()-t0:.1f}s')
print('Periodo:', df['Timestamp'].min(), '—', df['Timestamp'].max())
print('Valores faltantes:', int(df.isna().sum().sum()))
print('Duplicados:', int(df.duplicated().sum()))
print('Proporción de transacciones positivas:', f"{df['Is Laundering'].mean():.6%}")
display(df.head())

assert df['Timestamp'].notna().all(), 'Existen timestamps inválidos.'
assert set(df['Is Laundering'].unique()).issubset({0,1})

Filas: 5,078,345 | Columnas: 13 | Carga: 6.7s
Periodo: 2022-09-01 00:00:00 — 2022-09-18 16:18:00
Valores faltantes: 0
Duplicados: 9
Proporción de transacciones positivas: 0.101943%


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,sender_id,dest_id
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.340088,US Dollar,3697.340088,US Dollar,Reinvestment,0,10::8000EBD30,10::8000EBD30
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.010000,US Dollar,0.010000,US Dollar,Cheque,0,3208::8000F4580,1::8000F5340
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.570312,US Dollar,14675.570312,US Dollar,Reinvestment,0,3209::8000F4670,3209::8000F4670
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.969971,US Dollar,2806.969971,US Dollar,Reinvestment,0,12::8000F5030,12::8000F5030
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.968750,US Dollar,36682.968750,US Dollar,Reinvestment,0,10::8000F5200,10::8000F5200
